In [1]:
!pip3 install fastmcp langchain langchain_mcp_adapters langgraph langchain_openai --break-system-packages

In [2]:
import socket
import asyncio
from fastmcp import FastMCP, Client
import os

PORT = 8000

def make_dir():
    if os.path.exists("path"):
        print("✓ Path directory already exists")
    else:
        print("✗ Path directory doesn't exist - creating it...")
        os.makedirs("path")
        print("✓ Path directory created")

def test_port(port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('127.0.0.1', port))
            return False
        except socket.error:
            return True

f"Port {PORT} is available: {not test_port()}"

/opt/homebrew/lib/python3.13/site-packages/pydantic/plugin/_loader.py:50: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/opt/homebrew/lib/python3.13/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  warnings.warn(


'Port 8000 is available: True'

In [3]:
def print_stream_info(read, write, _sid, verbose=False):
    """Print information about the read/write streams and session ID.
    """
    if verbose:
        print("READ (receives FROM server):")
        print(read)
        print()
        
        print("WRITE (sends TO server):")
        print(write)
        print()
        
        print("SESSION ID:")
        print(_sid())

##### Tools

If you are familiar with agentic principles, you probably know about tools and their functionality. Let's take a quick look at tooling in **LangChain**.


In [4]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
   """Multiply two numbers."""
   return a * b

print(multiply.name)
print(multiply.description)
print(multiply.args)

print("What is 2 x 3?")
print("Answer: " + str(multiply.invoke({"a": 2, "b": 3})))

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
What is 2 x 3?
Answer: 6


### Creating a Calculator MCPServer
MCP servers behave like functions that AI agents can call to perform actions using. Let's define and configure a basic MCP server.

- **FastMCP** creates the server instance with a name and instructions
- `@mcp.tool` decorator registers functions as callable tools
- `@mcp.prompt` decorator creates reusable prompt templates
- `@mcp.resource` decorator exposes data resources with URI patterns


This creates a complete MCP server named **CalculatorMCPServer** with mathematical tools (add, subtract), a document reading resource, and a code review prompt template.

##### FastMCP Object 

The first step is to create an MCP server object ```mcp``` for the calculator. Like before, the object will be used to control the server. The parameters are:

- `name`: "CalculatorMCPServer" - the name of the server  
- `instructions`: "This server provides mathematical calculation tools. Call ```add()``` and ```subtract()``` to perform basic arithmetic operations." - this describes the functionality of the server


In [5]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="CalculatorMCPServer",
    instructions="""
        This server provides data analysis tools.
        Call get_average() to analyze numerical data.
    """)
print('mcp object',mcp)

mcp object FastMCP('CalculatorMCPServer')


##### Tools
 
 **Tools** (Active Capabilities)
    - Functions that AI agents can call to perform actions
    - Similar to LangChain tools but networked and discoverable

We define the MCP tools `add` and `subtract` - these will be called on the MCP object. 


In [6]:
@mcp.tool
def add(a: int, b: int) -> int:
    """
    Add two integers together.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The sum of `a` and `b`.

    Example:
        >>> add(3, 5)
        8
    """
    return a + b


@mcp.tool
def subtract(a: int, b: int) -> int:
    """
    Subtract one integer from another.

    Args:
        a (int): The number to subtract from.
        b (int): The number to subtract.

    Returns:
        int: The result of `a - b`.

    Example:
        >>> subtract(10, 4)
        6
    """
    return a - b

##### Resources

Resources are data sources (like files) that AI can read through URI-style addresses.

Use `@mcp.resource` to expose them.  
A pattern like `file:///endpoint/{name}` lets the path parameter select which resource to load.

URIs are for identification, so name paths clearly based on content/type.

In this example, we define two resources:
1. a prewritten message template  
2. the contents of a real file

In [8]:
@mcp.resource("file:///endpoint/{name}")
def return_template_document(name: str) -> str:
    """Read a document by name"""
    return f"Document contents of {name}"

[03/15/26 20:17:03] WARNING  Component already exists: template:file:///endpoint/{name}@      ]8;id=805421;file:///opt/homebrew/lib/python3.13/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=129605;file:///opt/homebrew/lib/python3.13/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

In [9]:
make_dir()

✓ Path directory already exists


In [10]:
%%capture
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/aNE__JjH4DLNEibuNpfDlg/examples.txt
!wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/tfoeGPInNoajVS0DSohdVg/README.txt

In [11]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name: str) -> str:
    """Read a document by name from the path directory"""
    try:
        # Read from the actual file system path
        with open(f"path/{name}", "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Document '{name}' not found in path directory"
    except Exception as e:
        return f"Error reading document: {str(e)}"

##### Prompts 

Prompts are consistent, reusable templates that can be called for simple, repetitive tasks. They capture domain expertise in a structured way, so instead of reinventing instructions each time, the AI can rely on a proven pattern.


In [12]:
@mcp.prompt(title="Code Review")
def review_code(code: str) -> str:
    return f"Please review this code:\n\n{code}"

### Creating a Client — In-Memory Transport

After defining MCP tools, you can test them with a client.  
For the simplest setup, use **in-memory transport**: client and server run in the same Python process, so no separate transport object is needed.

You still call tools through client methods, but communication is asynchronous, so use `async`/`await` to interact with the server.

In [13]:
from fastmcp import  Client

client = Client(mcp)
print(f"client: {client}")

client: <fastmcp.client.client.Client object at 0x11ac31d30>


##### Tools

To test an MCP server, use a client and invoke tools through `call_tool()`.  
Even with in-memory transport, this still follows client-server MCP communication (not direct local object calls).

Because server responses are asynchronous:
- `async with client`: manages connection lifecycle automatically  
- `await`: waits for server responses without blocking the process

In [14]:
async def call_add_tool(a: int, b: int):
    async with client:
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [16]:
response = await call_add_tool(4, 5)
response

CallToolResult(content=[TextContent(type='text', text='9', annotations=None, meta=None)], structured_content={'result': 9}, meta=None, data=9, is_error=False)

In [17]:
# The actual answer/data
print("\nResult Data .data :")
print(response.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(response.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(response.structured_content)  


Result Data .data :
9

Content (as text):
9

Structured Content:
{'result': 9}


### Question

Write a function to call the subtract tool with parameters a=4 and b=5, then execute it and print the result. What will be the output?

In [18]:
async def call_subtract_tool(a: int, b: int):
    async with client:
        result = await client.call_tool("subtract", {"a": a, "b": b})
        return result

response = await call_subtract_tool(4, 5)
print(response.content[0].text)

-1


The client object has useful attributes to get information about the server. Like before, we use asynchronous operations to communicate with the server. We use ```await client.list_tools()``` to get all available tools. Then we loop through and print each tool's name and description.

In [19]:
async with client:
    tools = await client.list_tools()
    print("Available tools:")
    for tool in tools:
        print(f"- {tool.name}: {tool.description}")

Available tools:
- add: Add two integers together.

Args:
    a (int): The first integer.
    b (int): The second integer.

Returns:
    int: The sum of `a` and `b`.

Example:
    >>> add(3, 5)
    8
- subtract: Subtract one integer from another.

Args:
    a (int): The number to subtract from.
    b (int): The number to subtract.

Returns:
    int: The result of `a - b`.

Example:
    >>> subtract(10, 4)
    6


In [20]:
tool_obj = tools[0]
print(tool_obj)

name='add' title=None description='Add two integers together.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The sum of `a` and `b`.\n\nExample:\n    >>> add(3, 5)\n    8' inputSchema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'} outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True} icons=None annotations=None meta={'fastmcp': {'tags': []}} execution=None


```Input Schema``` Defines how the tool takes arguments in a standardized JSON format, necessary because data is transmitted over the internet/network between client and server.

In [21]:
input_schema = tool.inputSchema
print(input_schema)

{'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}


```Output Schema```: JSON structure defining what the tool returns (data type and structure of the response) - necessary because responses are transmitted over the network as JSON; tells clients (and LLMs) what to expect back. MCP doesn't always expose this since some tools perform actions (like saving to database) where the LLM only needs simple confirmation, not detailed output data.


In [22]:
output_schema = tool.outputSchema
print(output_schema )

{'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}


##### Resources

We'll call the `read_resource` method in an asynchronous context manager wrapped in a function. The parameter for reading the resource is the URI with our configured `name`.


In [23]:
async def call_resource(name):
    async with client:
        result = await client.read_resource(f"file:///endpoint/{name}")
        return result

In [24]:
response = await call_resource("README.txt")
print(response[0].text)

Document contents of README.txt


In [25]:
async def call_resource2(name):
    async with client:
        result = await client.read_resource(f"file://endpoint2/{name}")
        return result

In [26]:
response = await call_resource2("README.txt")
response = await call_resource2("random.txt")
resource = response[0]

In [27]:
print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/random.txt
mimeType: text/plain
meta:     None
text:     Document 'random.txt' not found in path directory


##### Prompts

We'll also create a function to call the prompt method to review code. The only parameter is the code content to be reviewed.


In [28]:
async def call_prompt(code):
    async with client:

        result = await client.get_prompt("review_code", {"code": code})
        return result

In [29]:
response = await call_prompt("CODE TO BE REVIEWED")
message=response.messages[0]
print(f"Prompt Role:{message.role}")
print(f"Prompt Content:{message.content.text}")

Prompt Role:user
Prompt Content:Please review this code:

CODE TO BE REVIEWED


### HTTP Transport MCP Servers
HTTP transport allows MCP servers to run as web services that clients connect to via URLs. This is ideal for remote servers, cloud deployments, or when you want multiple clients to share the same server instance.


First, check if the port is available (True). If available, run the following cell to start the MCP server on that port. If the port is unavailable, change the PORT variable and rerun the server. This requirement is just for JupyterLab.

In [30]:
f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

##### Starting an HTTP MCP Server

Start the MCP server as a background HTTP service with:

```python
mcp.run_http_async(port=PORT)
```

This keeps the server running and exposes `/mcp` for JSON-RPC requests.

In Jupyter, wrap it with `asyncio.create_task()` so the server runs concurrently and doesn’t block other notebook cells.

In [ ]:
#server_task = asyncio.create_task(server_task_)
asyncio.create_task(mcp.run_http_async(port=PORT))
print(f"HTTP MCP Server started in background on port {PORT}")

HTTP MCP Server started in background on port 8000




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 3.1.0                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      CalculatorMCPServer, 3.1.0                  │
│                  🚀 Deploy free: https://fastmcp.cloud                       │
│                          

[03/15/26 20:26:09] INFO     Starting MCP server 'CalculatorMCPServer' with transport 'http' on    ]8;id=166144;file:///opt/homebrew/lib/python3.13/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=985515;file:///opt/homebrew/lib/python3.13/site-packages/fastmcp/server/mixins/transport.py#273\273]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [69003]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


### HTTP Transport  and Client
#####  HTTP Transport
Now we'll create an MCP client that uses `HTTP` transport to communicate with the server. This creates a transport object that tells the MCP Client how to communicate with the MCP server over HTTP.


In [32]:
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport

transport_http = StreamableHttpTransport(url=f"http://127.0.0.1:{PORT}/mcp")
http_client = Client(transport_http)
print('http_client',http_client )

http_client <fastmcp.client.client.Client object at 0x11d658f50>


In [33]:
async def test_client_http(client: Client, a: int, b: int)->int:
    async with client:  
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [34]:
response = await test_client_http(http_client, 4, 5)
print(response.content[0].text)

INFO:     127.0.0.1:57710 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57711 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57712 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57713 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57714 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57715 - "DELETE /mcp HTTP/1.1" 200 OK
9


### Exercise 
Write an async function called get_tool_list that takes a Client object as a parameter and returns a list of all available tools from the MCP server. Then call this function with the http_client and print the results.


In [35]:
async def get_tool_list(client: Client):
    async with client:
        abstools = await client.list_tools()
        return  abstools
        
tool_list=await get_tool_list(http_client)
print(tool_list)

INFO:     127.0.0.1:57726 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57727 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57728 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57729 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57730 - "DELETE /mcp HTTP/1.1" 200 OK
[Tool(name='add', title=None, description='Add two integers together.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The sum of `a` and `b`.\n\nExample:\n    >>> add(3, 5)\n    8', inputSchema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None), Tool(name='subtract', title=None, description='Subtract one integer from another.\n\nArgs:\n    a (int): The num

##### LangChain Tools with HTTP MCP Servers

Turning MCP tools into LangChain tools is straightforward, but the HTTP setup needs an extra adapter layer (`langchain-mcp-adapters`).  
`ClientSession` works with `StreamableHttpTransport` to communicate with the MCP server.

Before using `StreamableHttpTransport`, import:
`ClientSession`, `load_mcp_tools`, `create_react_agent`, and your LLM.

The ```streamablehttp_client``` function connects you to an MCP server over HTTP. When you call it, it opens a connection and gives you back three things that let you communicate with the server:

```read:``` Receives messages from the server

```write:``` Sends messages to the server

```_sid:``` Returns your unique connection ID

In [38]:
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from mcp import ClientSession
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],  # or HF_TOKEN
    base_url="https://router.huggingface.co/v1",
    model="Qwen/Qwen3.5-397B-A17B:novita",
    temperature=0.0,
)

In [39]:
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    print_stream_info(read, write, _sid, verbose=True)

READ (receives FROM server):
MemoryObjectReceiveStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

WRITE (sends TO server):
MemoryObjectSendStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

SESSION ID:
None


/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/contextlib.py:109: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)


In [40]:
async with streamablehttp_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    async with ClientSession(read, write) as session:
        await session.initialize()
        
        # Load LangChain tools from the LIVE MCP session
        tools = await load_mcp_tools(session)
        
        # Build the agent WHILE the session is still open
        agent = create_react_agent(
            model=llm,
            tools=tools,
        )
        agent_response = await agent.ainvoke({"messages": "Use the add tool to add 2 and 1 and let me know if you used a tool."})

INFO:     127.0.0.1:57819 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57820 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57821 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57822 - "POST /mcp HTTP/1.1" 200 OK


/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_69003/201918465.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


INFO:     127.0.0.1:57824 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57828 - "DELETE /mcp HTTP/1.1" 200 OK


In [41]:
print(agent_response['messages'][-1].content)

Yes, I used the add tool to add 2 and 1. The result is 3.


### STDIO Transport in MCP Servers

With STDIO transport, the client launches the MCP server as a child process and communicates via `stdin/stdout`.  
Unlike HTTP (which connects to a running URL), STDIO is process-to-process.

In Jupyter, run the server from a separate `.py` file, not as a regular notebook function.  
The key requirement is `mcp.run()` in `__main__`, which starts listening on `stdin/stdout`.

So you place the server code in a string (for example, `server_content`) and write it to a file before launching it.

In [42]:
server_content = '''
from fastmcp import FastMCP

mcp = FastMCP(
    name="CalculatorMCPServer",  # Fixed: added missing comma
    instructions="""
        This server provides data analysis tools.
        Call add() to add two numbers.
    """
)

@mcp.tool
def add(a: int, b: int) -> int:
    """Adds two integer numbers together."""
    return a + b

@mcp.tool
def subtract(a:int, b:int) -> int:
    """Subtracts b from a"""
    return a - b

@mcp.resource("file://documents/{name}")
def read_document(name: str) -> str:
    """Read a document by name"""
    return "Document contents of {name"

@mcp.prompt(title="Code Review")
def review_code(code: str) -> str:
    return f"Please review this code: {code}"

if __name__ == "__main__":
    mcp.run()
'''

In [43]:
with open('stdio_server.py', 'w') as f:
    f.write(server_content)

print("Created corrected stdio_server.py file")

Created corrected stdio_server.py file


##### Configuring STDIO Transport

Now let's create a StdioTransport to communicate over STDIO transport by defining the command and arguments to launch the server. This will start the server using our ```stdio_server.py``` file.

The ```StdioTransport``` function specifies the command and arguments to launch the server:

```command="python"``` tells the system to use the Python interpreter

```args=["stdio_server.py"]``` provides the script to execute


In [46]:
transport_stdio = StdioTransport(
    command="python",
    args=["stdio_server.py"]
)

The above configuration method is common when adding MCP servers. For example, some clients (e.g., Cursor, Claude Desktop) have `mcp.json` files that allow you to configure both **Remote** (HTTP) and **Local** (STDIO) MCP servers.

`mcp.json`:
```json
{
    "mcpServers": {
        "local_stdio_server_name": {
            "command": "npx/python/uv/uvx",
            "args": ["some absolute/relative path", "some install argument"]
        },
        "remote_http_server_name": {
            "url": "mcp_server_url"
        }
    }
}

```

##### Creating a STDIO Transport Client

Now we'll create an MCP client that uses our `StdioTransport` to communicate with the server. We'll test it the same way we called tools with the in-memory server above.


In [47]:
stdio_transport_client = Client(transport_stdio)
print(stdio_transport_client)

In [48]:
async def test_client(client: Client, a: int, b: int):
    async with client:
        tools = await client.list_tools()
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

response = await test_client(stdio_transport_client, 4, 5)
print(response.content[0].text)

9


### LangChain Tools

The process for using tools in ```STDIO``` is similar to HTTP - you just replace ```streamablehttp_client``` with ```stdio_client```. Both return ```(read, write)``` streams that you use to create a ClientSession. After initializing the session, we use ```load_mcp_tools``` to convert the MCP tools into LangChain-compatible format.


In [49]:
# In an async context manager, convert and print the tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
# Configure the STDIO server parameters
server_params = StdioServerParameters(
    command="python",
    args=["stdio_server.py"],
)

In [51]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        agent = create_react_agent(
            model=llm,
            tools=tools,
        )

        agent_response = await agent.ainvoke(
            {"messages": "Use the add tool to add 2 and 1 ."}
        )

print(agent_response['messages'][-1].content)

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_69003/442978679.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


The result of adding 2 and 1 is 3.


##### Multiple MCP Servers

`langchain-mcp-adapters` supports connecting to multiple MCP servers and loading tools from all of them.

Use `MultiServerMCPClient` to connect to both STDIO and HTTP servers.  
Setup is similar to earlier client transport configs, just combined into one multi-server client.

In [52]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

client = MultiServerMCPClient(
    {
        "stdio-client": {
            "command": "python",
            "args": ["stdio_server.py"],
            "transport": "stdio"
        },
        "http-client": {
            "url": f"http://127.0.0.1:{PORT}/mcp",
            "transport": "streamable_http"
        }
    }
)

In [53]:
tools = await client.get_tools()
[tool.name for tool in tools]

INFO:     127.0.0.1:57930 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57931 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57932 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57933 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57934 - "DELETE /mcp HTTP/1.1" 200 OK


/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/contextlib.py:109: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)


['add', 'subtract', 'add', 'subtract']

##### Create an Agent

Now connect a LangGraph agent to your MCP client and let the LLM use MCP tools.

`MultiServerMCPClient` simplifies this by handling server connections and tool loading in one place, so you don’t need manual `ClientSession` setup or nested async contexts.

Steps:
- Configure your LLM (for example, `gpt-5-nano`)  
- Pass the LLM and tools from `MultiServerMCPClient` into the agent  
- Invoke with a prompt that explicitly asks the agent to use tools (otherwise it may answer from base knowledge)

In [54]:
agent = create_react_agent(
    model=llm,
    tools=tools
)
agent_response = await agent.ainvoke({"messages": "whats 8 + 7? use tools"})

/var/folders/td/rtvtvvjn3x1gfqswnc7bb6hm0000gn/T/ipykernel_69003/3797354216.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


INFO:     127.0.0.1:57949 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57950 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57951 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57952 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57953 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57954 - "DELETE /mcp HTTP/1.1" 200 OK


/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/contextlib.py:109: DeprecationWarning: Use `streamable_http_client` instead.
  self.gen = func(*args, **kwds)


In [55]:
for i in agent_response['messages']:
    if isinstance(i, HumanMessage):
        message_type = "HUMAN"
    elif isinstance(i, AIMessage):
        message_type = "AI"
    elif isinstance(i, ToolMessage):
        message_type = "TOOL"
    else:
        message_type = "OTHER"

    if i.content == '':
        i.content = "tool call"
    
    print(f"[{message_type}] {i.content}")

[HUMAN] whats 8 + 7? use tools
[AI] tool call
[TOOL] [{'type': 'text', 'text': '15', 'id': 'lc_686406eb-ae56-4573-bef3-22b5e03fdf46'}]
[AI] 8 + 7 = 15
